In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
 masterPipelineRunID = '1234'

In [0]:
configSilverDF = spark.read \
  .format("jdbc") \
  .option("url", f"jdbc:sqlserver://insuranceserver2002.database.windows.net:1433;database=insuarnce") \
  .option("dbtable", "insurance.bronze_to_silver_config") \
  .option("user",'user') \
  .option("password", 'Admin1234') \
  .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
  .load()

In [0]:
columnMappingDF = spark.read \
  .format("jdbc") \
  .option("url", f"jdbc:sqlserver://insuranceserver2002.database.windows.net:1433;database=insuarnce") \
  .option("dbtable", "insurance.column_mapping") \
  .option("user",'user') \
  .option("password", 'Admin1234') \
  .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
  .load()
columnMappingDF.createOrReplaceTempView("fieldMappingView")  #Creating the Temperory View for the session


In [0]:
display(columnMappingDF)

In [0]:
#Dataframe to list ---> .collect()
#creating Column Mapping Dictionary
def create_column_mapping_dict(ObjectName,columnMappingDF,df):
    source_cols = columnMappingDF.filter(col("SOURCE_TABLE")==f'{ObjectName}').select("SOURCE_COLUMN").rdd.flatMap(lambda x: x).collect()
    target_cols =  columnMappingDF.filter(col("SOURCE_TABLE")==f'{ObjectName}').select("TARGET_COLUMN").rdd.flatMap(lambda x: x).collect()
    column_mapping_dict = dict(zip(source_cols, target_cols))
    for i,j in column_mapping_dict.items():
        df = df.withColumnRenamed(i,j)
    return column_mapping_dict,df


In [0]:
def create_column_mapping_dict(ObjectName, columnMappingDF, df):

    source_cols = [r["SOURCE_COLUMN"] for r in
                   columnMappingDF.filter(col("SOURCE_TABLE") == ObjectName)
                   .select("SOURCE_COLUMN")
                   .collect()]

    target_cols = [r["TARGET_COLUMN"] for r in
                   columnMappingDF.filter(col("SOURCE_TABLE") == ObjectName)
                   .select("TARGET_COLUMN")
                   .collect()]

    column_mapping_dict = dict(zip(source_cols, target_cols))

    for i, j in column_mapping_dict.items():
        df = df.withColumnRenamed(i, j)

    return column_mapping_dict, df

In [0]:

# Handling Nulls
def handle_nulls(bronzeDF, columnMappingDF, ObjectName):

    print(f"Null handling for {ObjectName}")

    nullMapping = columnMappingDF.filter(
        (col("VALUE").isNotNull()) &
        (col("SOURCE_TABLE") == ObjectName)
    ).select("TARGET_COLUMN", "VALUE").collect()

    # Create dictionary {column : value}
    nullHandlingDict = {row["TARGET_COLUMN"]: row["VALUE"] for row in nullMapping}

    # Apply fillna
    for column, value in nullHandlingDict.items():
        bronzeDF = bronzeDF.fillna({column: value})

    return bronzeDF

In [0]:
from pyspark.sql.functions import col

def data_type_casting(bronzeDF, columnMappingDF, ObjectName):

    print(f"Datatype casting for {ObjectName}")

    dtypeMapping = columnMappingDF.filter(
        col("SOURCE_TABLE") == ObjectName
    ).select("TARGET_COLUMN", "TARGET_DATATYPE").collect()

    # Create dictionary {column : datatype}
    dataTypeDict = {row["TARGET_COLUMN"]: row["TARGET_DATATYPE"] for row in dtypeMapping}

    # Apply datatype casting
    for column, dtype in dataTypeDict.items():
        bronzeDF = bronzeDF.withColumn(column, col(column).cast(dtype))

    return bronzeDF

In [0]:
def getColumnsForMerge(sourceName, targetTableName, sourceAlias="s", targetAlias="t", skipCols=None):
    if skipCols is None:
        skipCols = ["Added_On", "Added_By", "Modified_On", "Modified_By"]

    rows = spark.sql(f"""
                SELECT TARGET_COLUMN
                FROM fieldMappingView
                WHERE lower(SOURCE_SYSTEM) = lower('{sourceName}')
                AND lower(TARGET_TABLE) = lower('{targetTableName}')
                AND TARGET_COLUMN NOT IN (
                    'ADDED_BY',
                    'ADDED_ON',
                    'MODIFIED_BY',
                    'MODIFIED_ON')
                            ORDER BY ID, TARGET_COLUMN
                        """).collect()
    
    if not rows:
        raise ValueError(f"No mappings found for {sourceName} -> {targetTableName}")

    src_columns = []
    tgt_columns = []
    merge_pairs = []

    for r in rows:
        dest_col = r["TARGET_COLUMN"]
        src_col  = r["TARGET_COLUMN"]

        if dest_col in skipCols:
            continue

        src_columns.append(f"{sourceAlias}.{src_col}")
        tgt_columns.append(dest_col)
        merge_pairs.append(f"{targetAlias}.{dest_col} = {sourceAlias}.{src_col}")

    
    # audit_fields = ["Added_On", "Added_By", "Modified_On", "Modified_By"]
    target_columns_full = tgt_columns 

    merge_pairs.append(f"{targetAlias}.ADDED_ON = {sourceAlias}.ADDED_ON")
    merge_pairs.append(f"{targetAlias}.ADDED_BY = {sourceAlias}.ADDED_BY")

    sourceColumn = ", ".join(src_columns)
    targetColumn = ", ".join(target_columns_full)
    mergeColumnStatement = ", ".join(merge_pairs)
    targetColumn = targetColumn + ", ADDED_BY, ADDED_ON, MODIFIED_BY, MODIFIED_ON"
    insertSourceColumn = sourceColumn + ", s.ADDED_BY, s.ADDED_ON, s.MODIFIED_BY, s.MODIFIED_ON"

    return sourceColumn,insertSourceColumn, targetColumn, mergeColumnStatement

In [0]:
def load_data_into_silver(sourceName,objectName ,targetTableName, bronzeDF, operationType, keyColumnName,targetCatalogName,targetSchemaName,isFullLoad,masterPipelineRunID):
    bronzeDF.createOrReplaceTempView("tempView")
    srcColumns,insertSourceColumn,targetColumns,mergeColumnStatement = getColumnsForMerge(sourceName, targetTableName, sourceAlias="s", targetAlias="t", skipCols=None)
    if operationType.lower() == "upsert":
            if isFullLoad == 1:
                print(f"FULL LOAD STARTED FOR {objectName}")
                delete_query = spark.sql(f"DELETE FROM {targetCatalogName}.{targetSchemaName}.{targetTableName}")
                insert_query = spark.sql(f"""
                                    INSERT INTO {targetCatalogName}.{targetSchemaName}.{targetTableName} ({targetColumns})
                                    SELECT {insertSourceColumn}
                                    FROM tempView s
                                    """)
                df_history = spark.sql(f"describe history {targetCatalogName}.{targetSchemaName}.{targetTableName} limit 1").first()
                rowsInserted = df_history["operationMetrics"]["numOutputRows"]
                print("Number of Rows Inserted :", rowsInserted)
            else:
                print(f"Merge Operation Started on {targetTableName}")
                mergeQuery = f"""
                MERGE INTO {targetCatalogName}.{targetSchemaName}.{targetTableName} t
                USING tempView s ON s.{keyColumnName}=t.{keyColumnName}
                WHEN MATCHED THEN UPDATE
                SET {mergeColumnStatement},Modified_On = CURRENT_TIMESTAMP, Modified_By = '{masterPipelineRunID}'
                WHEN NOT MATCHED THEN INSERT ({targetColumns}) VALUES ({srcColumns},CURRENT_TIMESTAMP,'{masterPipelineRunID}', CURRENT_TIMESTAMP, '{masterPipelineRunID}')
                """
                print("Executing merge query")
                spark.sql(mergeQuery)
    

In [0]:
try:
    for row in configSilverDF.collect():

        ObjectName = row['OBJECT_NAME']
        sourceName = 'api'
        targetTableName = row['SILVER_TABLE_NAME']
        operationType = row['OPERATION_TYPE']
        keyColumnName = row['KEY_COLUMN_NAME']
        targetCatalogName = row['SILVER_CATALOG_NAME']
        targetSchemaName = row['SILVER_SCHEMA_NAME']
        isFullLoad = row['IS_HISTORICAL']

        sourceCatalogName = row['BRONZE_CATALOG_NAME']
        sourceSchemaName = row['BRONZE_SCHEMA_NAME']
        sourceTableName = row['BRONZE_TABLE_NAME']

        # Read bronze table
        bronzeDF = spark.read.table(f"{sourceCatalogName}.{sourceSchemaName}.{sourceTableName.lower()}")

        # Get source column list (WITHOUT RDD)
        sourceColumnList = [
            r["SOURCE_COLUMN"] for r in
            columnMappingDF.filter(col("SOURCE_TABLE") == ObjectName)
            .select("SOURCE_COLUMN")
            .collect()
        ]

        # Select only required columns
        bronzeDF = bronzeDF.select(*sourceColumnList)

        # Create column mapping dictionary and rename columns
        objectdict, bronzeDF = create_column_mapping_dict(
            ObjectName,
            columnMappingDF,
            bronzeDF
        )

        # Handle null values
        bronzeDF = handle_nulls(
            bronzeDF,
            columnMappingDF,
            ObjectName
        )

        # Data type casting
        bronzeDF = data_type_casting(
            bronzeDF,
            columnMappingDF,
            ObjectName
        )

        # Load into Silver
        load_data_into_silver(
            sourceName,
            ObjectName,
            targetTableName,
            bronzeDF,
            operationType,
            keyColumnName,
            targetCatalogName,
            targetSchemaName,
            isFullLoad,
            masterPipelineRunID
        )

except Exception as e:
    print(f"Error occurred while processing object {ObjectName}: {e}")
    raise

In [0]:
%sql
SHOW TABLES IN bronze;